In [ ]:
from datasets import load_dataset

In [ ]:
HF_TOKEN = "" # Add your hf token here.

In [ ]:
ds = load_dataset("rojagtap/bookcorpus")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

books_large_p1.txt:   0%|          | 0.00/2.52G [00:00<?, ?B/s]

books_large_p2.txt:   0%|          | 0.00/2.10G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/74004228 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer, PreTrainedTokenizerFast
import tokenizers

In [ ]:
# Load the tokenizer from the local 'hopper.json' file using the tokenizers library directly
# Then convert it to a PreTrainedTokenizerFast object.
tokenizer_backend = tokenizers.Tokenizer.from_file("/content/hopper.json")
tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer_backend, model_input_names=["input_ids", "attention_mask"])

Please provide the `input_text` you would like to tokenize. Once you provide the text, I will tokenize it and count the number of tokens.

In [ ]:
input_text = "SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24." # @param {type: "string"}

In [ ]:
# Tokenize the input text
tokens = tokenizer.tokenize(input_text)

# Count the number of tokens
num_tokens = len(tokens)

print(f"The input text: '{input_text}'")
print(f"The tokens are: {tokens}")
print(f"Number of tokens: {num_tokens}")

The input text: 'SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24.'
The tokens are: ['seb', '##i', 'study', 'finds', '9', '##3', '%', 'of', 'individual', 'f', '&', 'o', 'traders', 'made', 'losses', 'between', 'f', '##y', '##22', 'and', 'f', '##y', '##2', '##4', '.']
Number of tokens: 25


Now, let's add the token 'FY' to the tokenizer's vocabulary and re-tokenize the input text.

In [ ]:
# Add 'FY' as a new token to the tokenizer's vocabulary
num_added_toks = tokenizer.add_tokens('FY')
print(f"Number of tokens added: {num_added_toks}")

# Retokenize the input text with the updated tokenizer
tokens_with_fy = tokenizer.tokenize(input_text)

# Count the number of tokens
num_tokens_with_fy = len(tokens_with_fy)

print(f"\nInput text: '{input_text}'")
print(f"Tokens with 'FY' added: {tokens_with_fy}")
print(f"Number of tokens with 'FY' added: {num_tokens_with_fy}")

Number of tokens added: 1

Input text: 'SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24.'
Tokens with 'FY' added: ['seb', '##i', 'study', 'finds', '9', '##3', '%', 'of', 'individual', 'f', '&', 'o', 'traders', 'made', 'losses', 'between', 'fy', '22', 'and', 'fy', '24', '.']
Number of tokens with 'FY' added: 22


Let's load the 'bert-base-uncased' and 'gpt2' tokenizers and examine their special tokens.

In [ ]:
# Load the BERT tokenizer
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Get and print BERT's special tokens
print("BERT Special Tokens:")
print(bert_tokenizer.all_special_tokens)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BERT Special Tokens:
['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']


In [ ]:
# Load the GPT-2 tokenizer
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Get and print GPT-2's special tokens
print("\nGPT-2 Special Tokens:")
print(gpt2_tokenizer.all_special_tokens)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


GPT-2 Special Tokens:
['<|endoftext|>']


First, let's load the `imdb` dataset (excluding the 'unsupervised' split).

In [ ]:
# Load the 'imdb' dataset, skipping the 'unsupervised' split
imdb_dataset = load_dataset("stanfordnlp/imdb", split=['train', 'test'])

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Next, I'll define a function to tokenize the dataset and count the total number of tokens for each tokenizer. This process can be time-consuming, so I will process it in batches.

In [ ]:
def count_tokens(dataset_split, tokenizer):
    total_tokens = 0
    for example in dataset_split:
        # Ensure the input is a string for tokenization
        text = str(example['text'])
        # Tokenize and count tokens for each example
        total_tokens += len(tokenizer.tokenize(text))
    return total_tokens

tokenizer_counts = {}

print("Tokenizing with hopper tokenizer...")
# Tokenize with hopper tokenizer
hopper_total_tokens = count_tokens(imdb_dataset[0], tokenizer) + count_tokens(imdb_dataset[1], tokenizer)
tokenizer_counts['hopper'] = hopper_total_tokens
print(f"Hopper tokenizer total tokens: {hopper_total_tokens:,}")

print("\nTokenizing with BERT tokenizer...")
# Tokenize with bert-base-uncased tokenizer
bert_total_tokens = count_tokens(imdb_dataset[0], bert_tokenizer) + count_tokens(imdb_dataset[1], bert_tokenizer)
tokenizer_counts['bert-base-uncased'] = bert_total_tokens
print(f"BERT tokenizer total tokens: {bert_total_tokens:,}")

print("\nTokenizing with GPT-2 tokenizer...")
# Tokenize with gpt2 tokenizer
gpt2_total_tokens = count_tokens(imdb_dataset[0], gpt2_tokenizer) + count_tokens(imdb_dataset[1], gpt2_tokenizer)
tokenizer_counts['gpt2'] = gpt2_total_tokens
print(f"GPT-2 tokenizer total tokens: {gpt2_total_tokens:,}")

Tokenizing with hopper tokenizer...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (718 > 512). Running this sequence through the model will result in indexing errors


Hopper tokenizer total tokens: 15,353,417

Tokenizing with BERT tokenizer...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


BERT tokenizer total tokens: 15,416,058

Tokenizing with GPT-2 tokenizer...
GPT-2 tokenizer total tokens: 14,812,432


Now, let's sort the tokenizers by their total token count in ascending order and present the result.

In [ ]:
# Sort the tokenizers by token count in ascending order
sorted_tokenizers = sorted(tokenizer_counts.items(), key=lambda item: item[1])

print("\nTokenizers ordered by total token count (ascending):")
for name, count in sorted_tokenizers:
    print(f"{name}: {count:,} tokens")

# Prepare the output string as requested
order_string = ""
for name, _ in sorted_tokenizers:
    if name == 'hopper':
        order_string += '1'
    elif name == 'bert-base-uncased':
        order_string += '2'
    elif name == 'gpt2':
        order_string += '3'

print(f"\nOutput in the requested format (1=hopper, 2=bert-base-uncased, 3=gpt2): {order_string}")


Tokenizers ordered by total token count (ascending):

Output in the requested format (1=hopper, 2=bert-base-uncased, 3=gpt2): 


In [ ]:
markdown_content = """
# Notebook Summary: Tokenization for Deep Learning Practice

This notebook explores fundamental concepts and practical implementations of text tokenization, a crucial step in preparing text data for deep learning models. It covers custom tokenizers, pre-trained tokenizers, and a method for comparing their efficiency based on total token counts.

## Theory and Math Behind Tokenization

1.  **What is Tokenization?**
    *   Tokenization is the process of breaking down raw text into smaller units called "tokens." These tokens are then mapped to numerical IDs that deep learning models can process.
    *   Different tokenizers use varying algorithms (e.g., Byte-Pair Encoding, WordPiece) which result in different token sequences and counts for the same input text.

2.  **Subword Tokenization:**
    *   Many modern tokenizers employ subword tokenization, breaking words into smaller meaningful units (e.g., "unfriendliness" -> "un", "friend", "li", "ness"). This strategy helps handle rare words (reducing out-of-vocabulary issues) and keeps the overall vocabulary size manageable.
    *   The example `FY22` tokenized as `['fy', '22']` demonstrates how subword units are formed, contrasting with a more granular tokenization like `['f', '##y', '##22']` before the `FY` token was explicitly added to the vocabulary.

3.  **Types of Tokenizers:**
    *   **Custom Tokenizers:** These are built and configured for specific datasets or tasks. They often use a backend library (like `tokenizers`) and can be wrapped with `transformers.PreTrainedTokenizerFast` to integrate with the Hugging Face ecosystem, providing a consistent API.
    *   **Pre-trained Tokenizers:** These are readily available tokenizers associated with popular models (e.g., BERT, GPT-2). They are loaded using `transformers.AutoTokenizer.from_pretrained` and come with pre-defined vocabularies and tokenization rules.

4.  **Vocabulary and Special Tokens:**
    *   **Vocabulary:** The complete set of unique tokens a tokenizer recognizes.
    *   **Adding Tokens:** The vocabulary can be dynamically extended (`tokenizer.add_tokens('FY')`) to include new significant units. This can lead to more efficient tokenization by reducing the number of tokens required to represent certain phrases.
    *   **Special Tokens:** These are reserved tokens used for specific model functionalities:
        *   `[UNK]` (Unknown): Represents words not in the tokenizer's vocabulary.
        *   `[SEP]` (Separator): Used to separate different segments of text.
        *   `[PAD]` (Padding): Used to make sequences of varying lengths uniform.
        *   `[CLS]` (Classifier): Often placed at the beginning of a sequence for classification tasks (in BERT).
        *   `[MASK]` (Mask): Used in masked language modeling tasks.

5.  **Dataset Handling:**
    *   Datasets are often divided into splits (e.g., 'train', 'test') for training and evaluation. For comprehensive analysis, it's crucial to process all relevant splits of the dataset.
    *   The notebook uses `datasets.load_dataset` to efficiently load large datasets like `imdb`.

6.  **Token Counting as a Metric:**
    *   Comparing the total number of tokens generated by different tokenizers for the same dataset provides insight into their "verbosity" or "compression" capabilities. A tokenizer that produces fewer tokens for the same input might be considered more efficient in terms of sequence length, which can impact computational cost and model capacity.

## Code Implementation

### 1. Environment Setup and Library Imports

```python
from datasets import load_dataset
from transformers import AutoTokenizer, PreTrainedTokenizerFast
import tokenizers

# Set Hugging Face token (though unauthenticated warnings were observed in some cells)
HF_TOKEN = "hf_..."
```

### 2. Custom Tokenizer Loading and Wrapping

```python
# Load a custom tokenizer from a JSON file
tokenizer_backend = tokenizers.Tokenizer.from_file("/content/hopper.json")
# Wrap it with PreTrainedTokenizerFast for compatibility
tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer_backend, model_input_names=["input_ids", "attention_mask"])
```

### 3. Basic Tokenization and Counting

```python
input_text = "SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24."

# Tokenize the input text
tokens = tokenizer.tokenize(input_text)
num_tokens = len(tokens)

print(f"The tokens are: {tokens}")
print(f"Number of tokens: {num_tokens}")
```

### 4. Extending Vocabulary and Re-tokenization

```python
# Add 'FY' as a new token to the tokenizer's vocabulary
num_added_toks = tokenizer.add_tokens('FY')

# Retokenize to see the effect
tokens_with_fy = tokenizer.tokenize(input_text)
num_tokens_with_fy = len(tokens_with_fy)

print(f"Tokens with 'FY' added: {tokens_with_fy}")
print(f"Number of tokens with 'FY' added: {num_tokens_with_fy}")
```

### 5. Loading Pre-trained Tokenizers and Examining Special Tokens

```python
# Load BERT tokenizer
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("BERT Special Tokens:")
print(bert_tokenizer.all_special_tokens)

# Load GPT-2 tokenizer
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
print("GPT-2 Special Tokens:")
print(gpt2_tokenizer.all_special_tokens)
```

### 6. Dataset Loading

```python
# Load the 'imdb' dataset, specifically 'train' and 'test' splits
imdb_dataset = load_dataset("stanfordnlp/imdb", split=['train', 'test'])
```

### 7. Token Counting Function and Execution

```python
def count_tokens(dataset_split, tokenizer):
    total_tokens = 0
    for example in dataset_split:
        text = str(example['text'])
        total_tokens += len(tokenizer.tokenize(text))
    return total_tokens

tokenizer_counts = {}

# Count tokens for hopper tokenizer across train and test splits
hopper_total_tokens = count_tokens(imdb_dataset[0], tokenizer) + count_tokens(imdb_dataset[1], tokenizer)
tokenizer_counts['hopper'] = hopper_total_tokens

# Count tokens for BERT tokenizer
bert_total_tokens = count_tokens(imdb_dataset[0], bert_tokenizer) + count_tokens(imdb_dataset[1], bert_tokenizer)
tokenizer_counts['bert-base-uncased'] = bert_total_tokens

# Count tokens for GPT-2 tokenizer
gpt2_total_tokens = count_tokens(imdb_dataset[0], gpt2_tokenizer) + count_tokens(imdb_dataset[1], gpt2_tokenizer)
tokenizer_counts['gpt2'] = gpt2_total_tokens

print(f"Hopper tokenizer total tokens: {hopper_total_tokens:,}")
print(f"BERT tokenizer total tokens: {bert_total_tokens:,}")
print(f"GPT-2 tokenizer total tokens: {gpt2_total_tokens:,}")
```

### 8. Sorting and Presenting Results

```python
# Sort tokenizers by total token count in ascending order
sorted_tokenizers = sorted(tokenizer_counts.items(), key=lambda item: item[1])

print("\nTokenizers ordered by total token count (ascending):")
for name, count in sorted_tokenizers:
    print(f"{name}: {count:,} tokens")

# Generate specific order string based on original request
order_string = ""
for name, _ in sorted_tokenizers:
    if name == 'hopper':
        order_string += '1'
    elif name == 'bert-base-uncased':
        order_string += '2'
    elif name == 'gpt2':
        order_string += '3'

print(f"\nOutput in the requested format (1=hopper, 2=bert-base-uncased, 3=gpt2): {order_string}")
```
"""

with open('/content/summary.md', 'w') as f:
    f.write(markdown_content)

print("Summary saved to /content/summary.md")

Summary saved to /content/summary.md
